In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('minioDelta')\
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.0.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262"
        ])
    )\
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    ) \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    ) \
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://host.docker.internal:9000"
    ) \
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "minioadmin"
    ) \
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "minioadmin"
    ) \
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    ) \
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    ) \
    .getOrCreate()




In [44]:
df = spark.read.parquet(
    "s3a://bronze/employees/"
)

df.show()

# df.write.format("delta")\
#     .mode("append")\
#     .save("/home/jovyan/work/lakehouse/bronze/employees")

+-----------+-----------+----------+---------+--------------+----------+----------+-------+--------------+----------+-------------+--------------------+------------+---------------+
|EMPLOYEE_ID| FIRST_NAME| LAST_NAME|    EMAIL|  PHONE_NUMBER| HIRE_DATE|    JOB_ID| SALARY|COMMISSION_PCT|MANAGER_ID|DEPARTMENT_ID| ingestion_timestamp|source_table|       batch_id|
+-----------+-----------+----------+---------+--------------+----------+----------+-------+--------------+----------+-------------+--------------------+------------+---------------+
|        100|     Steven|      King|    SKING|1.515.555.0100|2013-06-17|   AD_PRES|24000.0|          NULL|      NULL|         90.0|2026-05-13 15:16:...|   employees|20260513_151636|
|        101|      Neena|      Yang|    NYANG|1.515.555.0101|2015-09-21|     AD_VP|17000.0|          NULL|     100.0|         90.0|2026-05-13 15:16:...|   employees|20260513_151636|
|        102|        Lex|    Garcia|  LGARCIA|1.515.555.0102|2011-01-13|     AD_VP|17000.0

In [48]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('overwriteSchema', 'true')\
    .save('s3a://lakehouse/bronze/employees')

In [5]:
import sys 
sys.path.append('/work/')

In [49]:
from pyspark.sql import DataFrame
from extract.config import TABLES

def parquet_to_delta(table_name):
    parquet_path = f's3a://bronze/{table_name}/'
    delta_path = f's3a://lakehouse/bronze/{table_name}'

    df = spark.read.parquet(parquet_path)
    df.write.format('delta')\
        .mode('overwrite')\
        .option('overwriteSchema', 'true')\
        .save(delta_path)
    

for table in TABLES:
    parquet_to_delta(table)
    

In [42]:
spark.read.format("delta").load(
    "s3a://lakehouse/bronze/employees"
).printSchema()

root
 |-- EMPLOYEE_ID: long (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- EMAIL: string (nullable = true)
 |-- PHONE_NUMBER: string (nullable = true)
 |-- HIRE_DATE: string (nullable = true)
 |-- JOB_ID: string (nullable = true)
 |-- SALARY: double (nullable = true)
 |-- COMMISSION_PCT: double (nullable = true)
 |-- MANAGER_ID: double (nullable = true)
 |-- DEPARTMENT_ID: double (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- source_table: string (nullable = true)



In [ ]:
import os

os.makedirs('/work/lakehouse/silver', exist_ok=True)

employees_df = spark.read.format('delta').load(
    '/work/lakehouse/bronze/employees'
)

departments_df = spark.read.format('delta').load(
    '/work/lakehouse/bronze/departments'
)

jobs_df = spark.read.format('delta').load(
    '/work/lakehouse/bronze/jobs'
)

employees_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/employees')

departments_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/departments')

jobs_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/jobs')




In [24]:
# rest of the local files pushed to bronze minio s3 bucket
countries_df = spark.read.format('delta').load(
    '/work/lakehouse/bronze/countries'
)

locations_df = spark.read.format('delta').load(
    '/work/lakehouse/bronze/locations'
)

regions_df = spark.read.format('delta').load(   
    '/work/lakehouse/bronze/regions'
)

countries_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/countries')

locations_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/locations')

regions_df.write.format('delta') \
    .mode('append')\
    .save('s3a://lakehouse/bronze/regions')


In [50]:
employees_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/employees"
)

departments_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/departments"
)

jobs_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/jobs"
)

countries_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/countries"
)

locations_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/locations"
)

regions_df = spark.read.format("delta").load(
    "s3a://lakehouse/bronze/regions"
)

In [18]:
employees_df.printSchema()

root
 |-- EMPLOYEE_ID: long (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- EMAIL: string (nullable = true)
 |-- PHONE_NUMBER: string (nullable = true)
 |-- HIRE_DATE: string (nullable = true)
 |-- JOB_ID: string (nullable = true)
 |-- SALARY: double (nullable = true)
 |-- COMMISSION_PCT: double (nullable = true)
 |-- MANAGER_ID: double (nullable = true)
 |-- DEPARTMENT_ID: double (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- source_table: string (nullable = true)



In [51]:
from pyspark.sql import functions as F

employees_silver_df = employees_df.alias('e')\
    .join(
        departments_df.alias('d'),
        F.col('e.department_id') == F.col('d.department_id'),
        'left'
    )\
    .join(
        jobs_df.alias('j'),
        F.col('e.job_id') == F.col('j.job_id'),
        'left'
    )\
    .join(
        locations_df.alias('l'),
        F.col('d.location_id') == F.col('l.location_id'),
        'left'
    )\
    .join(
      countries_df.alias('c'),
      F.col('l.country_id') == F.col('c.country_id'),
      'left'  
    )\
    .join(
        regions_df.alias('r'),
        F.col('c.region_id') == F.col('r.region_id'),
        'left'
    )\
    .select(
        F.col('e.employee_id'),
        F.trim(F.col('e.first_name')).alias('first_name'),
        F.trim(F.col('e.last_name')).alias('last_name'),
        F.lower(F.col('e.email')).alias('email'),
        F.col('e.salary').cast('double'),
        F.col('d.department_name'),
        F.col('j.job_title'),

        F.col('l.city'),
        F.col('c.country_name'),
        F.col('r.region_name'),
        F.col('e.hire_date'),
        F.year(F.col('e.hire_date')).alias('hire_year'),

        F.round(
            F.months_between(
                F.current_date(),
                F.col('e.hire_date')
            ) / 12, 1
        ).alias('tenure_years'),

        F.when(
            F.col('e.salary') < 20000,
            'low'
        ).when(
            F.col('e.salary').between(20000, 60000),
            'mid'
        ).otherwise('high').alias('salary_band'),

        F.current_timestamp().alias('processing_timestamp'),
        F.col('e.ingestion_timestamp'),
        F.col('e.source_table')
    )


employees_silver_df.show()

+-----------+-----------+----------+---------+-------+---------------+--------------------+---------+--------------------+-----------+----------+---------+------------+-----------+--------------------+--------------------+------------+
|employee_id| first_name| last_name|    email| salary|department_name|           job_title|     city|        country_name|region_name| hire_date|hire_year|tenure_years|salary_band|processing_timestamp| ingestion_timestamp|source_table|
+-----------+-----------+----------+---------+-------+---------------+--------------------+---------+--------------------+-----------+----------+---------+------------+-----------+--------------------+--------------------+------------+
|        100|     Steven|      King|    sking|24000.0|      Executive|           President|  Seattle|United States of ...|   Americas|2013-06-17|     2013|        12.9|        mid|2026-05-13 10:07:...|2026-05-13 15:16:...|   employees|
|        101|      Neena|      Yang|    nyang|17000.0|  

In [52]:
# employees_silver_df.columns
employees_silver_df.write.format('delta')\
    .mode('overwrite')\
    .save('s3a://lakehouse/silver/employees_enriched')

In [53]:
employee_hierarchy_df = employees_df.alias('e1')\
    .join(
        employees_df.alias('e2'),
        F.col('e1.manager_id') == F.col('e2.employee_id'),
        'left'
    )\
    .select(
        F.col('e1.employee_id'),
        F.concat_ws(
            " ",
            F.col('e1.first_name'),
            F.col('e1.last_name')
        ).alias('employee_name'),
        F.col('e1.manager_id'),
        F.concat_ws(
            " ",
            F.col('e2.first_name'),
            F.col('e2.last_name')
        ).alias('manager_name')
    )

employee_hierarchy_df.show()
employee_hierarchy_df.write.format('delta')\
    .mode('overwrite') \
    .save('s3a://lakehouse/silver/employee_hierarchy')

+-----------+-----------------+----------+---------------+
|employee_id|    employee_name|manager_id|   manager_name|
+-----------+-----------------+----------+---------------+
|        100|      Steven King|      NULL|               |
|        101|       Neena Yang|     100.0|    Steven King|
|        102|       Lex Garcia|     100.0|    Steven King|
|        103|  Alexander James|     102.0|     Lex Garcia|
|        104|     Bruce Miller|     103.0|Alexander James|
|        105|   David Williams|     103.0|Alexander James|
|        106|    Valli Jackson|     103.0|Alexander James|
|        107|     Diana Nguyen|     103.0|Alexander James|
|        108|  Nancy Gruenberg|     101.0|     Neena Yang|
|        109|    Daniel Faviet|     108.0|Nancy Gruenberg|
|        110|        John Chen|     108.0|Nancy Gruenberg|
|        111|   Ismael Sciarra|     108.0|Nancy Gruenberg|
|        112|Jose Manuel Urman|     108.0|Nancy Gruenberg|
|        113|        Luis Popp|     108.0|Nancy Gruenber

In [54]:
employee_tenure_df = employees_df.select(
    F.col('employee_id'),
    F.concat_ws(
        " ",
        F.col('first_name'),
        F.col('last_name')
    ).alias('employee_name'),
    F.col('hire_date'),
    F.round(
        F.months_between(
            F.current_date(),
            F.col('hire_date')
        ) / 12, 
        1
    ).alias('tenure_years'),
    F.year(F.col('hire_date')).alias('hire_year'),
    F.quarter(F.col('hire_date')).alias('hire_quarter'),
    F.month(F.col('hire_date')).alias('hire_month'),
    F.col('salary'),
    F.when(
        F.col('salary') < 20000,
        'low'
    ).when(
        F.col('salary').between(20000, 60000),
        'mid'
    ).otherwise('high').alias('salary_band'),
    F.when(
        F.months_between(
            F.current_date(),
            F.col('hire_date')
        ) / 12 < 2,
        'New'
    ).when(
        F.months_between(
            F.current_date(),
            F.col('hire_date')
        ) / 12 < 5,
        'senior noob'
    ).otherwise('long term').alias('tenure_category'),
    F.current_timestamp().alias('processing_timestamp')
)

employee_tenure_df.write.format('delta')\
    .mode('overwrite')\
    .save('s3a://lakehouse/silver/employee_tenure_anals')

In [56]:
employees_silver_df = spark.read.format("delta").load(
    "s3a://lakehouse/silver/employees_enriched"
)

employees_silver_df.show()

+-----------+-----------+----------+---------+-------+---------------+--------------------+---------+--------------------+-----------+----------+---------+------------+-----------+--------------------+--------------------+------------+
|employee_id| first_name| last_name|    email| salary|department_name|           job_title|     city|        country_name|region_name| hire_date|hire_year|tenure_years|salary_band|processing_timestamp| ingestion_timestamp|source_table|
+-----------+-----------+----------+---------+-------+---------------+--------------------+---------+--------------------+-----------+----------+---------+------------+-----------+--------------------+--------------------+------------+
|        100|     Steven|      King|    sking|24000.0|      Executive|           President|  Seattle|United States of ...|   Americas|2013-06-17|     2013|        12.9|        mid|2026-05-13 10:08:...|2026-05-13 15:16:...|   employees|
|        101|      Neena|      Yang|    nyang|17000.0|  

In [58]:
from pyspark.sql.window import Window

gold_workforce_df = employees_silver_df.groupBy(
    'region_name', 'department_name'
).agg(
    F.count('*').alias('total_employees'),
    F.avg('salary').alias('avg_salary'),
    F.max('salary').alias('max_salary'),
    F.min('salary').alias('min_salary'),
)

gold_workforce_df.write.format('delta')\
    .partitionBy('region_name')\
    .mode('overwrite')\
    .save('s3a://lakehouse/gold/workforce_kpis')

# compensation kpis
gold_compensation_df = employees_silver_df.groupBy(
    'job_title'
).agg(
    F.avg('salary').alias('avg_salary'),
    F.max('salary').alias('max_salary'),
    F.min('salary').alias('min_salary'),
    F.count('*').alias('emp_count')

)
gold_compensation_df.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://lakehouse/gold/compensation_kpis")

# hiring patterns kpis
gold_hiring_df = employee_tenure_df.groupBy(
    'hire_year'
).agg(
    F.count('*').alias('total_hires')
).orderBy('hire_year')

win = Window.orderBy('hire_year')
gold_hiring_df = gold_hiring_df.withColumn(
    'prev_year_hires',
    F.lag('total_hires').over(win)
).withColumn(
    'growth_rate',
    ((F.col('total_hires') - F.col('prev_year_hires')) / F.col('prev_year_hires')) * 100 
)

gold_hiring_df.write.format('delta')\
    .mode('overwrite')\
    .save('s3a://lakehouse/gold/hiring_trend_kpis')
        


In [60]:
display(
    spark.read.format("delta").load(
        "s3a://lakehouse/gold/workforce_kpis"
    )
)

DataFrame[region_name: string, department_name: string, total_employees: bigint, avg_salary: double, max_salary: double, min_salary: double]

In [7]:
gold_workforce_df = spark.read.format("delta").load(
    "s3a://lakehouse/gold/workforce_kpis"
)

gold_compensation_df = spark.read.format("delta").load(
    "s3a://lakehouse/gold/compensation_kpis"
)

gold_hiring_df = spark.read.format("delta").load(
    "s3a://lakehouse/gold/hiring_trend_kpis"
)

import os 
os.makedirs(
    '/work/dashboard/data',
    exist_ok=True
)

gold_workforce_df.toPandas().to_parquet(
    "/work/dashboard/data/workforce_summary_kpis.parquet",
    index=False
)

gold_compensation_df.toPandas().to_parquet(
    "/work/dashboard/data/compensation_summary_kpis.parquet",
    index=False
)

gold_hiring_df.toPandas().to_parquet(
    "/work/dashboard/data/hiring_trend_kpis.parquet",
    index=False
)

